In [2]:
import stim
import re
import os
from pyzx import *
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network


In [3]:
import stim


stim.PauliString({1: "X", 3: "Y", 3: "X"})

stim.PauliString("+XZ")

In [4]:
def stim_qasm_comply(qasm: str) -> str:
    q = qasm
    q = re.sub(r'def\s+rx\(qubit q0\)\s*\{[^}]*\}\n+', '', q)
    q = re.sub(r'rx\s*\(\s*q\[(\d+)\]\s*\)\s*;', r'h q[\1];', q)
    q = re.sub(r'reset\s+q\[(\d+)\];', '', q)
    return q

tableau_5_code = stim.Tableau.from_stabilizers([
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"), 
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZXIXZ")
], allow_underconstrained=True)

tableau_bit_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZ_"),
    stim.PauliString("_ZZ"),
], allow_underconstrained=True)

print(tableau_bit_code)
# print(tableau_5_code)

input = 1  # Example: This means N = 4 qubits
out = 5

n = input + out


stabilizers = []

for i in range (out - input):
    stabilizers.append(stim.PauliString(f"Z{i}") * stim.PauliString(n))

# Loop must go from 0 to N-2
for i in range(out - input, n-1): 
    stabilizers.append(stim.PauliString(f"Z{i}*Z{i+1}") * stim.PauliString(n)) 

stabilizers.append(stim.PauliString("I" * (out - input))  + stim.PauliString("X" * (n - (out - input))))

print("Stabilizers:")
for s in stabilizers:
    print(s)

tableau = stim.Tableau.from_stabilizers(stabilizers)

state = stim.TableauSimulator()
state.set_state_from_stabilizers(stabilizers)
state.do_tableau(tableau_5_code, range(out - input + 1))

print(state.current_inverse_tableau().inverse())
t = state.current_inverse_tableau().inverse()

qasm = t.to_circuit(method="graph_state").to_qasm(open_qasm_version=3)

print(tableau_5_code)
pyzx_circ = Circuit.from_qasm(stim_qasm_comply(qasm))
draw(pyzx_circ  )
g = pyzx_circ.to_graph()
input_state = "0"*(n)
g.apply_state(input_state)
clifford_simp(g)
g.normalize()
g.auto_detect_io()
# draw(g, labels=True )

draw(g, labels=True )
# c = Circuit.from_graph(g)

# draw(c)


g = GraphState(g)
g.to_canonical_form()
draw(g)
t0 = tensorfy(g)

+-xz-xz-xz-
| ++ ++ ++
| XZ X_ X_
| _Z XZ X_
| __ _Z XZ
Stabilizers:
+Z_____
+_Z____
+__Z___
+___Z__
+____ZZ
+____XX
+-xz-xz-xz-xz-xz-xz-
| -+ ++ -+ ++ -- +-
| _X __ ZX _Z Z_ _Z
| _Z _X __ ZX _Z __
| XZ _Z XX __ XX _X
| _X _Z _Z _X XZ _X
| Z_ ZX ZZ ZZ __ __
| __ __ __ __ _Z ZX
+-xz-xz-xz-xz-xz-
| -+ ++ -+ ++ --
| _X __ ZX _Z Z_
| _Z _X __ ZX _Z
| XZ _Z XX __ XX
| _X _Z _Z _X XZ
| Z_ ZX ZZ ZZ __


In [5]:
nxg = nx.Graph()
bounds = []
for v in d["vertices"]:
    v_type = v["t"]
    if v["id"] in d["inputs"]:
        color = "green"
    else:
        color = "blue"
    if v_type != VertexType.BOUNDARY:
        nxg.add_node(v["id"], 
                    color=color, 
                    title=f"Type: {v_type.name}")
    else:
        bounds.append(v["id"])
        
for e in d["edges"]:
    vertexes = d["vertices"]
    edge_type = e[2]
    edge_color = "red" if edge_type == EdgeType.HADAMARD else "black"
    if e[0] not in bounds and e[1] not in bounds:
        nxg.add_edge(e[0], e[1])
        nxg[e[0]][e[1]]['color'] = edge_color

pos = nx.spring_layout(nxg, seed=42)  # nice spacing

node_colors = [nxg.nodes[n]['color'] for n in nxg.nodes()]
edge_colors = [nxg[u][v]['color'] for u,v in nxg.edges()]

# Draw the graph
plt.figure(figsize=(10,8))
nx.draw_networkx_nodes(nxg, pos, node_color=node_colors, node_size=700)
nx.draw_networkx_edges(nxg, pos, edge_color=edge_colors, width=2)
# nx.draw_networkx_labels(nxg, pos, font_size=10, font_color='white')

# Add title to the plot
plt.title("5 qubit code", fontsize=16, pad=20)

plt.axis('off')
plt.show()

net = Network(notebook=True, directed=False)
net.from_nx(nxg)


NameError: name 'd' is not defined

In [ ]:
print(t1)

[[[[[[-0.125+0.j -0.125+0.j]
     [ 0.125-0.j -0.125+0.j]]

    [[ 0.125-0.j -0.125+0.j]
     [ 0.125+0.j  0.125-0.j]]]


   [[[ 0.125-0.j -0.125+0.j]
     [-0.125+0.j -0.125+0.j]]

    [[ 0.125+0.j  0.125-0.j]
     [ 0.125-0.j -0.125+0.j]]]]



  [[[[ 0.125-0.j -0.125+0.j]
     [-0.125+0.j -0.125+0.j]]

    [[-0.125+0.j -0.125+0.j]
     [-0.125+0.j  0.125-0.j]]]


   [[[ 0.125-0.j  0.125-0.j]
     [-0.125+0.j  0.125-0.j]]

    [[ 0.125-0.j -0.125+0.j]
     [ 0.125-0.j  0.125-0.j]]]]]




 [[[[[ 0.125-0.j -0.125+0.j]
     [ 0.125+0.j  0.125-0.j]]

    [[-0.125+0.j -0.125+0.j]
     [ 0.125-0.j -0.125+0.j]]]


   [[[-0.125-0.j -0.125+0.j]
     [-0.125+0.j  0.125-0.j]]

    [[-0.125+0.j  0.125-0.j]
     [ 0.125-0.j  0.125-0.j]]]]



  [[[[ 0.125-0.j  0.125-0.j]
     [ 0.125-0.j -0.125+0.j]]

    [[-0.125+0.j  0.125-0.j]
     [ 0.125-0.j  0.125-0.j]]]


   [[[ 0.125-0.j -0.125+0.j]
     [ 0.125-0.j  0.125-0.j]]

    [[ 0.125-0.j  0.125-0.j]
     [-0.125+0.j  0.125-0.j]]]]]]


In [6]:

qasm_5qubit = tableau_5_code.to_circuit(method="elimination").to_qasm(open_qasm_version=3)

qasm_bit = tableau_bit_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZ_"),
    stim.PauliString("_ZZ"),
], allow_underconstrained=True).to_circuit(method="elimination").to_qasm(open_qasm_version=3)

pyzx_circ = Circuit.from_qasm(qasm_5qubit)
g = pyzx_circ.to_graph()
input_state = "0"*(4) + "/"*1
g.apply_state(input_state)
# g = GraphState(g)
# draw(g, labels=True)
# draw(g, labels=True )
# clifford_simp(g)
# g.normalize()
# # g.set_outputs(4)
# g.auto_detect_io()
# draw(g, labels=True)

# print(t2)


t1 = tensorfy(g)
clifford_simp(g)
g.normalize()
draw(g, labels=True )


# g.set_qubit(15, g.qubit(49) + 1)
# g.set_qubit(4, g.qubit(49) + 1)
# g.set_row(15, g.row(49))
# g.set_row(4, g.row(49)+ 1)

draw(g, labels=True)

g.auto_detect_io()

g = GraphState(g)
draw(g, labels=True)


g.to_canonical_form()
g = g.state_to_map()
t2 = tensorfy(g)
compare_tensors(t2, t0, preserve_scalar=False)

5 6
5 5


True

In [ ]:

qasm_5qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"), 
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZXIXZ")
], allow_underconstrained=True).to_circuit(method="elimination").to_qasm(open_qasm_version=3)

shor_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZIIIIIII"),
    stim.PauliString("ZIZIIIIII"),
    stim.PauliString("IIIZZIIII"),
    stim.PauliString("IIIZIZIII"),
    stim.PauliString("IIIIIIZZI"),
    stim.PauliString("IIIIIIZIZ"),
    stim.PauliString("XXXXXXIII"),
    stim.PauliString("IIIXXXXXX"),
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)

qasm_7qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("IIIXXXX"),
    stim.PauliString("IXXIIXX"),
    stim.PauliString("XIXIXIX"),
    stim.PauliString("IIIZZZZ"),
    stim.PauliString("IZZIIZZ"),
    stim.PauliString("ZIZIZIZ")
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3) 


In [ ]:

pyzx_circ = Circuit.from_qasm(qasm_5qubit)
g = pyzx_circ.to_graph()
input_state = "0"*(4) + "/"*1
g.apply_state(input_state)
d = to_universal_graph_representation(g)